# graphforest — synthetic fraud demo
Shows all six fraud types on a small synthetic dataset.

In [1]:
import pandas as pd
import numpy as np
from graphforest import DirectedGraphForestClassifier, GraphForestExplainer, FRAUD_TYPES
print('graphforest imported OK')
print('Fraud types:', FRAUD_TYPES)

graphforest imported OK
Fraud types: {0: 'normal', 1: 'friendly_fraud', 2: 'identity_fraud', 3: 'account_takeover', 4: 'mule_ring', 5: 'synthetic_identity', 6: 'merchant_collusion'}


In [2]:
df = pd.DataFrame({
    'sender':    ['A','B','C','D','E','F','B','D','G','H','I','B'],
    'receiver':  ['M1','M1','M2','M1','M3','M2','M2','M3','M1','M2','M3','M3'],
    'timestamp': list(range(1, 13)),
    'amount':    [100,500,80,300,60,90,600,400,120,75,200,700],
})
y = pd.Series([0,4,0,3,0,0,4,3,0,0,0,4])
print(df)
print('\nLabel counts:')
print(y.value_counts().rename(FRAUD_TYPES))

   sender receiver  timestamp  amount
0       A       M1          1     100
1       B       M1          2     500
2       C       M2          3      80
3       D       M1          4     300
4       E       M3          5      60
5       F       M2          6      90
6       B       M2          7     600
7       D       M3          8     400
8       G       M1          9     120
9       H       M2         10      75
10      I       M3         11     200
11      B       M3         12     700

Label counts:
normal              7
mule_ring           3
account_takeover    2
Name: count, dtype: int64


In [3]:
model = DirectedGraphForestClassifier(
    source_col='sender',
    target_col='receiver',
    timestamp_col='timestamp',
    amount_col='amount',
    n_estimators=100,
)
model.fit(df, y)
print('Model fitted. Classes:', model.classes_)

Model fitted. Classes: [0 3 4]


In [4]:
df['predicted_type'] = model.predict_fraud_type(df)
df['true_type'] = y.map(FRAUD_TYPES)
print(df[['sender','receiver','amount','true_type','predicted_type']])

   sender receiver  amount         true_type    predicted_type
0       A       M1     100            normal            normal
1       B       M1     500         mule_ring         mule_ring
2       C       M2      80            normal            normal
3       D       M1     300  account_takeover  account_takeover
4       E       M3      60            normal            normal
5       F       M2      90            normal            normal
6       B       M2     600         mule_ring         mule_ring
7       D       M3     400  account_takeover  account_takeover
8       G       M1     120            normal            normal
9       H       M2      75            normal            normal
10      I       M3     200            normal            normal
11      B       M3     700         mule_ring         mule_ring


In [5]:
import json
explainer = GraphForestExplainer(model)
explanation = explainer.explain(df, row_index=1)
print(json.dumps(explanation, indent=2, default=str))

{
  "transaction": {
    "sender": "B",
    "receiver": "M1",
    "timestamp": 2,
    "amount": 500,
    "predicted_type": "mule_ring",
    "true_type": "mule_ring"
  },
  "predicted_class": 4,
  "predicted_type": "mule_ring",
  "class_probas": {
    "normal": 0.0612,
    "account_takeover": 0.0444,
    "mule_ring": 0.8944
  },
  "graph_context": {
    "in_degree": 0.0,
    "out_degree": 3.0,
    "weighted_in_degree": 0.0,
    "weighted_out_degree": 1800.0,
    "pagerank": 0.0509,
    "hub_score": 0.5502,
    "authority_score": -0.0,
    "neighbor_fraud_rate": 0.0,
    "cycle_score": 0.0
  },
  "neighbor_context": {
    "neighborhood_risk_score": 0.0417,
    "nearby_fraud_nodes": [
      {
        "node": "D",
        "hops_away": 2
      }
    ],
    "hops_checked": 2
  },
  "tabular_rules": [],
  "graph_rules": [
    {
      "feature": "weighted_out_degree",
      "importance": 0.367,
      "value": 1800.0
    },
    {
      "feature": "out_degree",
      "importance": 0.33,
      "v

In [6]:
feat_imp = pd.Series(
    model._graph_rf.feature_importances_,
    index=model.graph_features
).sort_values(ascending=False)
print('Graph RF feature importances:')
print(feat_imp.round(4))

Graph RF feature importances:
weighted_out_degree    0.367
out_degree             0.330
hub_score              0.303
in_degree              0.000
weighted_in_degree     0.000
pagerank               0.000
authority_score        0.000
neighbor_fraud_rate    0.000
cycle_score            0.000
dtype: float64
